# Loading and Preprocessing

In [1]:
from data_loader import load_data

In [2]:
X_clinical,\
X_clinical_A,\
X_clinical_B,\
X_clinical_Delta,\
targets = load_data()

In [3]:
targets.head()

,surv_bestresponse_car,ae_summ_icans_v2,ae_summ_icans_highestgrade_v2,ae_summ_crs_v2,ae_summ_highestgrade_v2
FTC-UMCG-0001,1.0,1.0,1.0,1.0,1.0
FTC-UMCG-0003,4.0,0.0,0.0,1.0,1.0
FTC-UMCG-0004,4.0,1.0,3.0,1.0,2.0
FTC-UMCG-0005,2.0,1.0,2.0,1.0,1.0
FTC-UMCG-0006,1.0,0.0,0.0,1.0,2.0


In [4]:
datasets = {
    "Clinical": X_clinical,
    "Clinical_A": X_clinical_A,
    "Clinical_B": X_clinical_B,
    "Clinical_Delta": X_clinical_Delta
}

# Feature Selection
Dropping highly correlated clinical and radiomics features using the EDA analysis, as they add little to no new information.

## Highly Correlated Clinical Features  
We're using the EDA because it took the mixed feature types into account when calculating the correlations, and there are only a few features, so we could manually write them. So, it was more straight-forward than redoing it ourselves.

In [5]:
clinical_ftrs_to_drop = ['total_num_priortherapylines_fl',
                'tr_car_bridg_type.factor_None',
                'scr_weight',
                'scr_height',
                'indication_age_60',
                'cli_st_neutrophils',
                'indication_extran_invol',
                'indication_pri_refr',
                'indication_sec_refr'
                ]

In [6]:
for name, df in datasets.items():
    print(f"Before dropping features in {name}: {df.shape}")
    df.drop(columns=clinical_ftrs_to_drop, inplace=True)
    print(f"After dropping features in {name}: {df.shape}")

Before dropping features in Clinical: (85, 35)
After dropping features in Clinical: (85, 26)
Before dropping features in Clinical_A: (85, 134)
After dropping features in Clinical_A: (85, 125)
Before dropping features in Clinical_B: (85, 134)
After dropping features in Clinical_B: (85, 125)
Before dropping features in Clinical_Delta: (85, 134)
After dropping features in Clinical_Delta: (85, 125)


## Highly Correlated Radiomic Features
These are all numerical values, with high dimensionality. The EDA already showed a large number of highly correlated features, so we will write code to automatically keep only one of possibly multiple highly correlated features.

In [7]:
import numpy as np

def keep_uncorrelated_features(df, threshold=0.9):
    # Absolute correlation matrix
    corr = df.corr().abs()
    # Upper triangle (ignore diagonal and lower half), k=1 means we start from the first diagonal above the main diagonal
    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )
    # Find columns with any correlation above threshold
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any() and upper[col].dtype in [np.float64]]
    # Return reduced dataframe
    return df.drop(columns=to_drop)

# Usage
for name, df in datasets.items():
    print(f"Before dropping correlated features in {name}: {df.shape}")

    datasets[name] = keep_uncorrelated_features(df, threshold=0.9)
    print(f"After dropping correlated features in {name}: {datasets[name].shape}")

Before dropping correlated features in Clinical: (85, 26)
After dropping correlated features in Clinical: (85, 26)
Before dropping correlated features in Clinical_A: (85, 125)
After dropping correlated features in Clinical_A: (85, 62)
Before dropping correlated features in Clinical_B: (85, 125)
After dropping correlated features in Clinical_B: (85, 59)
Before dropping correlated features in Clinical_Delta: (85, 125)
After dropping correlated features in Clinical_Delta: (85, 71)


In [8]:
cat_cols = datasets['Clinical'].select_dtypes(exclude='number').columns.tolist()

In [9]:
# Using ColumnTransformer to apply different transformations to numeric and categorical columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Creating a ColumnTransformer to apply StandardScaler to numeric columns and leave categorical columns unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', 'passthrough', cat_cols)],
        remainder= StandardScaler()
    
)

In [10]:
from model_eval import result_viewer

# ICANS 0,1

In [11]:
y = targets['ae_summ_icans_v2']

In [12]:
y.value_counts()

ae_summ_icans_v2
1.0    48
0.0    37
Name: count, dtype: int64

In [13]:
clinical_ftrs = datasets['Clinical'].columns.tolist()
radio_delta = datasets['Clinical_Delta'].drop(clinical_ftrs, axis=1)

In [14]:
process_delta = StandardScaler()

In [15]:
icans_1 = result_viewer(process_delta, {'only_radio_delta': radio_delta}, y)

Results for Logistic Regression:


,only_radio_delta
test_roc_auc,0.3897 ± 0.1031
train_roc_auc,0.9453 ± 0.0139
test_precision,0.4834 ± 0.0514
train_precision,0.8647 ± 0.0396
test_recall,0.5200 ± 0.0532
train_recall,0.8852 ± 0.0276
test_f1,0.4991 ± 0.0402
train_f1,0.8742 ± 0.0249


Results for Decision Tree:


,only_radio_delta
test_roc_auc,0.4030 ± 0.1179
train_roc_auc,0.8050 ± 0.0191
test_precision,0.4405 ± 0.1209
train_precision,0.7696 ± 0.1040
test_recall,0.5178 ± 0.2543
train_recall,0.8382 ± 0.1767
test_f1,0.4653 ± 0.1819
train_f1,0.7772 ± 0.0630


Results for Random Forest:


,only_radio_delta
test_roc_auc,0.4749 ± 0.0764
train_roc_auc,0.9615 ± 0.0092
test_precision,0.5286 ± 0.0268
train_precision,0.7981 ± 0.0248
test_recall,0.7556 ± 0.1456
train_recall,0.9789 ± 0.0258
test_f1,0.6164 ± 0.0620
train_f1,0.8787 ± 0.0111


In [16]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6094 ± 0.1236,0.4929 ± 0.1157,0.5791 ± 0.1703,0.3893 ± 0.0783
train_roc_auc,0.8822 ± 0.0242,0.9899 ± 0.0083,0.9917 ± 0.0058,0.9982 ± 0.0018
test_precision,0.6594 ± 0.1180,0.6353 ± 0.1274,0.6745 ± 0.1797,0.4944 ± 0.0509
train_precision,0.8338 ± 0.0129,0.9640 ± 0.0261,0.9448 ± 0.0284,0.9946 ± 0.0108
test_recall,0.6622 ± 0.1814,0.7067 ± 0.1356,0.6467 ± 0.0980,0.4778 ± 0.1129
train_recall,0.8591 ± 0.0279,0.9688 ± 0.0194,0.9740 ± 0.0232,0.9738 ± 0.0235
test_f1,0.6509 ± 0.1207,0.6594 ± 0.0989,0.6487 ± 0.0978,0.4834 ± 0.0781
train_f1,0.8459 ± 0.0133,0.9664 ± 0.0225,0.9589 ± 0.0206,0.9840 ± 0.0156


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5653 ± 0.1128,0.5385 ± 0.0969,0.4640 ± 0.1036,0.4788 ± 0.0859
train_roc_auc,0.8021 ± 0.0141,0.8078 ± 0.0281,0.8141 ± 0.0167,0.8311 ± 0.0212
test_precision,0.6235 ± 0.0722,0.5744 ± 0.0930,0.5663 ± 0.0219,0.5412 ± 0.1097
train_precision,0.7835 ± 0.0332,0.7826 ± 0.0295,0.8170 ± 0.0770,0.8763 ± 0.0566
test_recall,0.6200 ± 0.1638,0.7489 ± 0.0839,0.7889 ± 0.1241,0.5511 ± 0.2078
train_recall,0.8385 ± 0.0714,0.8588 ± 0.0550,0.8489 ± 0.1421,0.7432 ± 0.1494
test_f1,0.6090 ± 0.0819,0.6487 ± 0.0872,0.6551 ± 0.0539,0.5264 ± 0.1118
train_f1,0.8077 ± 0.0345,0.8180 ± 0.0326,0.8181 ± 0.0431,0.7900 ± 0.0731


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6070 ± 0.1163,0.5756 ± 0.1036,0.6263 ± 0.0859,0.5394 ± 0.0873
train_roc_auc,0.9620 ± 0.0164,0.9719 ± 0.0161,0.9901 ± 0.0079,0.9858 ± 0.0088
test_precision,0.6105 ± 0.0593,0.5763 ± 0.0734,0.5793 ± 0.0148,0.5467 ± 0.0452
train_precision,0.7897 ± 0.0347,0.7731 ± 0.0164,0.8471 ± 0.0339,0.8431 ± 0.0425
test_recall,0.8511 ± 0.0923,0.7978 ± 0.1661,0.8600 ± 0.1744,0.8356 ± 0.1179
train_recall,0.9634 ± 0.0212,0.9895 ± 0.0211,0.9947 ± 0.0105,0.9842 ± 0.0129
test_f1,0.7107 ± 0.0710,0.6510 ± 0.0429,0.6834 ± 0.0550,0.6584 ± 0.0644
train_f1,0.8671 ± 0.0172,0.8677 ± 0.0060,0.9145 ± 0.0161,0.9074 ± 0.0219


## ICANS >=2

In [17]:
y = targets["ae_summ_icans_highestgrade_v2"]

In [18]:
y = y.astype('int8')

In [19]:
y[y<2] = 0
y[y>=2] = 1


In [20]:
y.value_counts()

ae_summ_icans_highestgrade_v2
0    52
1    33
Name: count, dtype: int64

In [21]:
icans_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6151 ± 0.1199,0.5715 ± 0.0777,0.6304 ± 0.0834,0.6393 ± 0.1411
train_roc_auc,0.8985 ± 0.0258,0.9887 ± 0.0085,0.9955 ± 0.0054,0.9931 ± 0.0078
test_precision,0.5362 ± 0.1828,0.4893 ± 0.1704,0.5450 ± 0.3211,0.4450 ± 0.2283
train_precision,0.7789 ± 0.0252,0.9449 ± 0.0401,0.9695 ± 0.0287,0.9763 ± 0.0311
test_recall,0.4714 ± 0.2504,0.4952 ± 0.2230,0.4095 ± 0.2725,0.5619 ± 0.3346
train_recall,0.6672 ± 0.0392,0.9094 ± 0.0510,0.9473 ± 0.0292,0.9328 ± 0.0593
test_f1,0.4815 ± 0.1985,0.4705 ± 0.1580,0.4331 ± 0.2367,0.4899 ± 0.2632
train_f1,0.7183 ± 0.0302,0.9267 ± 0.0447,0.9582 ± 0.0275,0.9535 ± 0.0432


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5135 ± 0.1017,0.5358 ± 0.1184,0.5931 ± 0.1033,0.5502 ± 0.1771
train_roc_auc,0.7474 ± 0.0355,0.8046 ± 0.0441,0.8151 ± 0.0345,0.8095 ± 0.0113
test_precision,0.3394 ± 0.1885,0.5365 ± 0.2433,0.4579 ± 0.1339,0.4605 ± 0.1938
train_precision,0.6400 ± 0.1115,0.7065 ± 0.0653,0.7391 ± 0.0708,0.6875 ± 0.0855
test_recall,0.4571 ± 0.3546,0.4238 ± 0.1048,0.4381 ± 0.2073,0.5143 ± 0.2355
train_recall,0.7969 ± 0.2250,0.6886 ± 0.0760,0.6895 ± 0.0439,0.8322 ± 0.1867
test_f1,0.3632 ± 0.2300,0.4269 ± 0.0243,0.4348 ± 0.1628,0.4756 ± 0.1975
train_f1,0.6742 ± 0.0623,0.6934 ± 0.0490,0.7121 ± 0.0492,0.7277 ± 0.0642


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6324 ± 0.1289,0.5372 ± 0.1384,0.6390 ± 0.0700,0.5908 ± 0.0822
train_roc_auc,0.9629 ± 0.0145,0.9887 ± 0.0049,0.9900 ± 0.0074,0.9957 ± 0.0065
test_precision,0.4000 ± 0.4899,0.0000 ± 0.0000,0.4000 ± 0.4899,0.2667 ± 0.3887
train_precision,0.9846 ± 0.0308,0.9875 ± 0.0250,1.0000 ± 0.0000,1.0000 ± 0.0000
test_recall,0.0905 ± 0.1170,0.0000 ± 0.0000,0.0952 ± 0.1313,0.0952 ± 0.1313
train_recall,0.4091 ± 0.0372,0.5006 ± 0.0951,0.6071 ± 0.0760,0.5772 ± 0.1167
test_f1,0.1460 ± 0.1858,0.0000 ± 0.0000,0.1500 ± 0.2000,0.1400 ± 0.1960
train_f1,0.5764 ± 0.0338,0.6580 ± 0.0872,0.7528 ± 0.0577,0.7246 ± 0.1001


# CRS 0,1

In [22]:
y = targets["ae_summ_crs_v2"]

In [23]:
y.value_counts()

ae_summ_crs_v2
1.0    79
0.0     6
Name: count, dtype: int64

In [24]:
crs_1 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5467 ± 0.3127,0.3233 ± 0.3291,0.4133 ± 0.2397,0.3633 ± 0.2488
train_roc_auc,0.9886 ± 0.0107,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_precision,0.9265 ± 0.0258,0.9241 ± 0.0247,0.9381 ± 0.0396,0.9248 ± 0.0251
train_precision,0.9520 ± 0.0139,0.9938 ± 0.0077,0.9815 ± 0.0115,0.9875 ± 0.0062
test_recall,0.9617 ± 0.0313,0.9242 ± 0.0466,0.8492 ± 0.2111,0.9367 ± 0.0559
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9435 ± 0.0248,0.9234 ± 0.0267,0.8709 ± 0.1344,0.9298 ± 0.0326
train_f1,0.9754 ± 0.0074,0.9969 ± 0.0039,0.9906 ± 0.0058,0.9937 ± 0.0031


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4933 ± 0.1210,0.4746 ± 0.0235,0.7183 ± 0.2101,0.4433 ± 0.0457
train_roc_auc,0.8250 ± 0.1004,0.8010 ± 0.1273,0.9407 ± 0.0447,0.7100 ± 0.0200
test_precision,0.9223 ± 0.0240,0.9249 ± 0.0251,0.9624 ± 0.0308,0.9211 ± 0.0239
train_precision,0.9628 ± 0.0073,0.9696 ± 0.0191,0.9875 ± 0.0063,0.9576 ± 0.0061
test_recall,0.8992 ± 0.0633,0.9367 ± 0.0396,0.9367 ± 0.0396,0.8867 ± 0.0914
train_recall,0.9810 ± 0.0064,1.0000 ± 0.0000,0.9969 ± 0.0063,1.0000 ± 0.0000
test_f1,0.9092 ± 0.0334,0.9302 ± 0.0250,0.9483 ± 0.0167,0.9009 ± 0.0502
train_f1,0.9718 ± 0.0039,0.9845 ± 0.0098,0.9921 ± 0.0000,0.9783 ± 0.0031


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4867 ± 0.1215,0.5567 ± 0.2853,0.5775 ± 0.3254,0.3392 ± 0.1943
train_roc_auc,0.9981 ± 0.0025,0.9956 ± 0.0032,0.9994 ± 0.0013,0.9953 ± 0.0028
test_precision,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9294 ± 0.0235
train_precision,0.9294 ± 0.0059,0.9322 ± 0.0070,0.9349 ± 0.0069,0.9377 ± 0.0056
test_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9633 ± 0.0129
train_f1,0.9634 ± 0.0031,0.9649 ± 0.0037,0.9664 ± 0.0037,0.9678 ± 0.0030


# CRS >=2

In [25]:
y = targets["ae_summ_highestgrade_v2"]
y = y.astype('int8')
y[y<2] = 0
y[y>=2] = 1

y.value_counts()

ae_summ_highestgrade_v2
0    55
1    30
Name: count, dtype: int64

In [26]:
crs_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6061 ± 0.1271,0.5606 ± 0.0883,0.5606 ± 0.1481,0.5727 ± 0.1269
train_roc_auc,0.8763 ± 0.0219,0.9924 ± 0.0064,0.9943 ± 0.0036,0.9987 ± 0.0014
test_precision,0.5124 ± 0.1885,0.4800 ± 0.1127,0.3867 ± 0.2640,0.4038 ± 0.1244
train_precision,0.7408 ± 0.0648,0.9496 ± 0.0312,0.9727 ± 0.0223,0.9917 ± 0.0167
test_recall,0.4000 ± 0.1700,0.3667 ± 0.1633,0.2667 ± 0.2000,0.4000 ± 0.1333
train_recall,0.5750 ± 0.0667,0.9333 ± 0.0333,0.8917 ± 0.0425,0.9333 ± 0.0425
test_f1,0.4378 ± 0.1684,0.4105 ± 0.1439,0.3135 ± 0.2260,0.3901 ± 0.1195
train_f1,0.6470 ± 0.0658,0.9411 ± 0.0283,0.9300 ± 0.0297,0.9609 ± 0.0216


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5076 ± 0.0669,0.6091 ± 0.0556,0.4758 ± 0.1459,0.5970 ± 0.1203
train_roc_auc,0.7583 ± 0.0277,0.8321 ± 0.0305,0.7825 ± 0.0301,0.8214 ± 0.0292
test_precision,0.2238 ± 0.1961,0.5222 ± 0.3251,0.5067 ± 0.4165,0.3436 ± 0.1943
train_precision,0.5784 ± 0.2974,0.8712 ± 0.1349,0.9274 ± 0.1130,0.8212 ± 0.1783
test_recall,0.1667 ± 0.1491,0.3333 ± 0.2357,0.2000 ± 0.1247,0.3667 ± 0.2867
train_recall,0.3833 ± 0.2831,0.5917 ± 0.2464,0.4833 ± 0.1333,0.6083 ± 0.2550
test_f1,0.1860 ± 0.1619,0.3527 ± 0.1958,0.2602 ± 0.1657,0.3315 ± 0.2033
train_f1,0.4266 ± 0.2346,0.6495 ± 0.1431,0.6133 ± 0.0797,0.6297 ± 0.1280


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6515 ± 0.1010,0.5909 ± 0.0698,0.6455 ± 0.1044,0.5636 ± 0.1540
train_roc_auc,0.9538 ± 0.0114,0.9634 ± 0.0050,0.9659 ± 0.0133,0.9938 ± 0.0079
test_precision,0.0000 ± 0.0000,0.0667 ± 0.1333,0.2667 ± 0.3887,0.2000 ± 0.4000
train_precision,1.0000 ± 0.0000,0.9581 ± 0.0567,0.9846 ± 0.0308,1.0000 ± 0.0000
test_recall,0.0000 ± 0.0000,0.0333 ± 0.0667,0.0667 ± 0.0816,0.0333 ± 0.0667
train_recall,0.2833 ± 0.1130,0.5583 ± 0.0425,0.5250 ± 0.0425,0.5583 ± 0.1106
test_f1,0.0000 ± 0.0000,0.0444 ± 0.0889,0.1016 ± 0.1260,0.0571 ± 0.1143
train_f1,0.4294 ± 0.1380,0.7048 ± 0.0440,0.6839 ± 0.0395,0.7099 ± 0.0948


# Outcome
CMR vs others

In [27]:
y = targets["surv_bestresponse_car"]

In [28]:
y.value_counts()

surv_bestresponse_car
1.0    63
2.0    12
4.0     8
0.0     2
Name: count, dtype: int64

Early Death: 0, CMR: 1, PMR:2, SD: 3, PD:4

In [29]:
y = y.astype('int8')

In [30]:
y[y != 1] = 0

In [31]:
y.value_counts()

surv_bestresponse_car
1    63
0    22
Name: count, dtype: int64

In [32]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.7228 ± 0.0718,0.5136 ± 0.1044,0.6351 ± 0.0582,0.6972 ± 0.0935
train_roc_auc,0.9057 ± 0.0206,0.9960 ± 0.0032,0.9982 ± 0.0018,0.9973 ± 0.0029
test_precision,0.7822 ± 0.0743,0.7630 ± 0.0272,0.8151 ± 0.0422,0.7918 ± 0.0708
train_precision,0.8549 ± 0.0267,0.9407 ± 0.0308,0.9621 ± 0.0234,0.9693 ± 0.0255
test_recall,0.8744 ± 0.1052,0.9205 ± 0.0718,0.7949 ± 0.1493,0.8397 ± 0.0550
train_recall,0.9325 ± 0.0206,0.9960 ± 0.0080,0.9960 ± 0.0080,0.9881 ± 0.0097
test_f1,0.8196 ± 0.0506,0.8334 ± 0.0399,0.7933 ± 0.0680,0.8144 ± 0.0600
train_f1,0.8918 ± 0.0199,0.9674 ± 0.0195,0.9786 ± 0.0129,0.9785 ± 0.0167


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5459 ± 0.0704,0.5359 ± 0.1080,0.5179 ± 0.0532,0.4900 ± 0.0974
train_roc_auc,0.7977 ± 0.0135,0.8253 ± 0.0513,0.8253 ± 0.0456,0.8371 ± 0.0577
test_precision,0.7658 ± 0.0338,0.7794 ± 0.0469,0.7388 ± 0.0488,0.7450 ± 0.0298
train_precision,0.8527 ± 0.0310,0.8826 ± 0.0530,0.8999 ± 0.0395,0.8750 ± 0.0408
test_recall,0.9372 ± 0.0580,0.8397 ± 0.0550,0.7462 ± 0.1651,0.8423 ± 0.1184
train_recall,0.9523 ± 0.0349,0.9369 ± 0.0520,0.9206 ± 0.0674,0.9762 ± 0.0232
test_f1,0.8423 ± 0.0383,0.8080 ± 0.0467,0.7365 ± 0.1095,0.7871 ± 0.0665
train_f1,0.8988 ± 0.0161,0.9061 ± 0.0149,0.9072 ± 0.0170,0.9217 ± 0.0140


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.7005 ± 0.1222,0.5833 ± 0.1321,0.6518 ± 0.1454,0.6190 ± 0.1013
train_roc_auc,0.9585 ± 0.0095,0.9735 ± 0.0134,0.9785 ± 0.0043,0.9969 ± 0.0026
test_precision,0.7500 ± 0.0228,0.7375 ± 0.0338,0.7382 ± 0.0270,0.7454 ± 0.0331
train_precision,0.7876 ± 0.0101,0.8240 ± 0.0246,0.8264 ± 0.0151,0.8158 ± 0.0183
test_recall,1.0000 ± 0.0000,0.9833 ± 0.0333,0.9846 ± 0.0308,0.9692 ± 0.0615
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.8569 ± 0.0151,0.8427 ± 0.0322,0.8432 ± 0.0191,0.8407 ± 0.0217
train_f1,0.8811 ± 0.0063,0.9033 ± 0.0147,0.9049 ± 0.0090,0.8985 ± 0.0112
